In [1]:
import torch
from diffusion.architectures.classifiers.mnist_classifier import ClassifierTrainer
from whar_datasets.support.getter import WHARDatasetID, get_dataset_cfg
from diffusion.sampleables.whar_sampleable import WHARSampleable, TrainValTest

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [3]:
cfg = get_dataset_cfg(WHARDatasetID.WISDM)

In [4]:
sampeable = WHARSampleable(
    cfg=cfg,
    scv_group_index=0,
    fold=TrainValTest.TRAIN,
    # transform=lambda x: x.unsqueeze(0),
)
val_sampeable = WHARSampleable(
    cfg=cfg,
    scv_group_index=0,
    fold=TrainValTest.VAL,
    # transform=lambda x: x.unsqueeze(0),
)

2025-10-16 16:30:34,956 - whar-datasets - INFO - Running DownloadingStep
2025-10-16 16:30:34,957 - whar-datasets - INFO - Checking hash for DownloadingStep
2025-10-16 16:30:34,957 - whar-datasets - INFO - Hash is up to date
2025-10-16 16:30:34,958 - whar-datasets - INFO - Running ParsingStep
2025-10-16 16:30:34,958 - whar-datasets - INFO - Checking hash for ParsingStep
2025-10-16 16:30:34,959 - whar-datasets - INFO - Hash is up to date
2025-10-16 16:30:34,959 - whar-datasets - INFO - Running WindowingStep
2025-10-16 16:30:34,959 - whar-datasets - INFO - Checking hash for WindowingStep
2025-10-16 16:30:34,960 - whar-datasets - INFO - Hash is up to date
2025-10-16 16:30:34,960 - whar-datasets - INFO - Loading windowing
2025-10-16 16:30:34,986 - whar-datasets - INFO - activity_ids from 0 to 5
2025-10-16 16:30:34,986 - whar-datasets - INFO - subject_ids from 0 to 35
2025-10-16 16:30:34,988 - whar-datasets - INFO - train: 17915 | test: 4493
2025-10-16 16:30:34,989 - whar-datasets - INFO - R

In [5]:
print(sampeable.shape)

(6, 32, 26)


In [6]:
from diffusion.architectures.classifiers.wisdm_classifier import WISDMClassifier


classifier = WISDMClassifier(in_c=sampeable.shape[0], num_classes=sampeable.num_classes)

trainer = ClassifierTrainer(
    classifier=classifier,
    train_data=sampeable,
    val_data=val_sampeable,
)

In [7]:
state_dict = trainer.train(
    num_epochs=15,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=500,
    validate=True,
)

2025-10-16 16:30:40,931 - flow-matching - INFO - Training model with size: 1.581 MiB
Epoch 0/15: 100%|██████████| 500/500 [01:08<00:00,  7.33it/s, train_loss=0.299545]
2025-10-16 16:31:49,677 - flow-matching - INFO - val loss: 1.0917017459869385, best val loss: 1.0917017459869385
Epoch 1/15: 100%|██████████| 500/500 [01:07<00:00,  7.40it/s, train_loss=0.090974]
2025-10-16 16:32:57,651 - flow-matching - INFO - val loss: 1.0245890617370605, best val loss: 1.0245890617370605
Epoch 2/15: 100%|██████████| 500/500 [01:07<00:00,  7.44it/s, train_loss=0.062279]
2025-10-16 16:34:05,279 - flow-matching - INFO - val loss: 1.0183287858963013, best val loss: 1.0183287858963013
Epoch 3/15: 100%|██████████| 500/500 [01:07<00:00,  7.44it/s, train_loss=0.047417]
2025-10-16 16:35:12,906 - flow-matching - INFO - val loss: 1.0330567359924316, best val loss: 1.0183287858963013
Epoch 4/15: 100%|██████████| 500/500 [01:07<00:00,  7.45it/s, train_loss=0.035238]
2025-10-16 16:36:20,502 - flow-matching - INFO -

In [8]:
torch.save(classifier.state_dict(), "./models/classifier.pt")